In [15]:
import os
from pathlib import Path
import glob


BASE_DIR = Path(".")
DATASET_1 = BASE_DIR / "Blueberry_1"
DATASET_2 = BASE_DIR / "Blueberry_2"

def get_dataset_files(directory):
    return {
        'images': list(directory.glob("**/*.jpg")),
        'labels_txt': list(directory.glob("**/*.txt")),
        'labels_json': list(directory.glob("**/*.json")),
        'labels_xml': list(directory.glob("**/*.xml"))
    }

blue_1_files = get_dataset_files(DATASET_1)
blue_2_files = get_dataset_files(DATASET_2)

print(f"Blueberry_1: {len(blue_1_files['images'])} images, {len(blue_1_files['labels_txt'])} labels")
print(f"Blueberry_2: {len(blue_2_files['images'])} images, {len(blue_2_files['labels_txt'])} txt, {len(blue_2_files['labels_json'])} json")

Blueberry_1: 466 images, 468 labels
Blueberry_2: 140 images, 140 txt, 140 json


# 5. Data Visualization (Exploratory Data Analysis)
Here we analyze the datasets to understand class distributions, image resolutions, and visually inspect some bounding boxes before preprocessing.

In [16]:
import random
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
import matplotlib.patches as patches
from collections import Counter

def get_class_distributions_and_resolutions(blue_1_files, blue_2_files): 
    classes = []
    widths = []
    heights = []
    fruit_counts = []
    
    datasets = [blue_1_files, blue_2_files]
    for dataset in datasets:
        for txt_file in dataset['labels_txt']:
            with open(txt_file, 'r') as f:
                lines = f.readlines()
                fruit_counts.append(len(lines))
                for line in lines:
                    parts = line.strip().split()
                    if parts:
                        try:
                            classes.append(int(parts[0]))
                        except ValueError:
                            pass

        for img_file in dataset['images']:
            try:
                with Image.open(str(img_file)) as img:
                    widths.append(img.width)
                    heights.append(img.height)
            except Exception:
                pass
                
    return classes, widths, heights, fruit_counts

print('Running EDA Data Collection...')
classes, widths, heights, fruit_counts = get_class_distributions_and_resolutions(blue_1_files, blue_2_files)
print(f'Analyzed {len(widths)} images and {len(classes)} annotations.\n')

def visualize_sample(dataset_files, dataset_name, cls_colors={0: 'lightgreen', 1:'firebrick', 3:'royalblue'}):
    images = dataset_files['images']
    sample_image_path = random.choice(images)
    label_path = sample_image_path.with_suffix('.txt')
    
    if not label_path.exists():
        print(f'Label missing for {sample_image_path}')
        return
        
    try:
        img_pil = Image.open(str(sample_image_path))
        img = np.array(img_pil)
        h, w, _ = img.shape
    except Exception:
        print('Failed to load image')
        return
    
    fig, ax = plt.subplots(1, figsize=(8, 8))
    ax.imshow(img)
    
    with open(label_path, 'r') as f:
        lines = f.readlines()
        
    for line in lines:
        parts = line.strip().split()
        if not parts: continue
        cls_id = int(parts[0])
        x_center, y_center, width, height = map(float, parts[1:])
        
        x = (x_center - width / 2) * w
        y = (y_center - height / 2) * h
        box_w = width * w
        box_h = height * h
        
        color = cls_colors.get(cls_id, 'yellow')
        rect = patches.Rectangle((x, y), box_w, box_h, linewidth=2, edgecolor=color, facecolor='none')
        ax.add_patch(rect)
        ax.text(x, y, str(cls_id), color=color, fontsize=12, weight='bold')
        
    plt.title(f"Sample from {dataset_name}: {sample_image_path.name}")
    plt.axis('off')
    plt.savefig('output_eda_sample.png')
    plt.close() # Close plot instead of unblocking show()


Running EDA Data Collection...
Analyzed 606 images and 45890 annotations.



In [17]:
fig, axs = plt.subplots(1, 3, figsize=(18, 5))

# Class Distribution
class_counts = Counter(classes)
axs[0].bar(class_counts.keys(), class_counts.values())
axs[0].set_title('Raw Class Distribution (Before Normalization)')
axs[0].set_xlabel('Class ID')
axs[0].set_ylabel('Count')
axs[0].set_xticks(list(class_counts.keys()))

# Image Resolutions
axs[1].scatter(widths, heights, alpha=0.5)
axs[1].set_title('Image Resolutions')
axs[1].set_xlabel('Width')
axs[1].set_ylabel('Height')

# Fruit Density
axs[2].hist(fruit_counts, bins=20)
axs[2].set_title('Number of Fruits per Image')
axs[2].set_xlabel('Number of Fruits')
axs[2].set_ylabel('Frequency')

plt.tight_layout()
plt.savefig('output_eda_charts.png')
plt.close()

print("Displaying a sample from Blueberry_1 (0=Unripe, 1=Ripe)")
visualize_sample(blue_1_files, 'Blueberry_1')

print("Displaying a sample from Blueberry_2 (0=Unripe, 3=Ripe)")
visualize_sample(blue_2_files, 'Blueberry_2')


Displaying a sample from Blueberry_1 (0=Unripe, 1=Ripe)
Label missing for Blueberry_1\train\images\P1000095-JPG_0_png.rf.a961224bd250bc00bdd91cdd1d27ea5a.jpg
Displaying a sample from Blueberry_2 (0=Unripe, 3=Ripe)


---

# 6. & 7. Data Preprocessing & Outlier Handling
Here we process both datasets, normalize the class IDs (0=Unripe, 1=Ripe), and filter out any extreme outliers (e.g., extremely large bounding boxes or missing labels) before we build our final unified dataset.

**Label Mapping:**
- **Blueberry_1**: Already `0` (Unripe) and `1` (Ripe). Left as-is.
- **Blueberry_2**: Uses `0` (Unripe/Unblue) and `3` (Ripe/Blue). Remapped `3` to `1`.

In [18]:
import shutil
import random

OUT_DIR = BASE_DIR / "unified_dataset"
if OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)

splits = ["train", "val", "test"]
for split in splits:
    (OUT_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
    (OUT_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

def process_and_filter_dataset(dataset_files, source_tag):
    valid_pairs = []
    outliers_removed = 0
    
    for img_path in dataset_files['images']:
        lbl_path = img_path.with_suffix('.txt')
        if not lbl_path.exists():
            continue
            
        # Read label to check for outliers and normalize classes
        with open(lbl_path, "r") as f:
            lines = f.readlines()
            
        if len(lines) == 0:
            outliers_removed += 1 # Empty label file = outlier/no fruit
            continue
            
        normalized_lines = []
        has_outlier = False
        for line in lines:
            parts = line.strip().split()
            if not parts: continue
            try:
                class_id = int(parts[0])
                _, _, width, height = map(float, parts[1:])
            except ValueError:
                continue
            
            # Outlier Detection: Extreme box sizes (>90% or <0.1% area)
            area = width * height
            if area > 0.90 or area < 0.001:
                has_outlier = True
                break

            # Normalization Logic
            if source_tag == "BB1":
                normalized_id = class_id
            elif source_tag == "BB2":
                if class_id == 0:
                    normalized_id = 0
                elif class_id == 3:
                    normalized_id = 1
                else:
                    continue # Drop invalid classes like 1 or 2 if they exist in BB2
            
            normalized_lines.append(f"{normalized_id} " + " ".join(parts[1:]) + "\n")
            
        # If there's an outlier box or if normalization killed all boxes, skip image
        if has_outlier or not normalized_lines:
            outliers_removed += 1
            continue
            
        valid_pairs.append({
            'img': img_path, 
            'lbl': lbl_path,
            'source': source_tag,
            'normalized_lines': normalized_lines
        })
        
    print(f"[{source_tag}] Kept {len(valid_pairs)} valid pairs. Removed {outliers_removed} outliers/invalid.")
    return valid_pairs

print("Processing datasets...")
bb1_valid = process_and_filter_dataset(blue_1_files, "BB1")
bb2_valid = process_and_filter_dataset(blue_2_files, "BB2")

all_valid_data = bb1_valid + bb2_valid
print(f"\nTotal clean, merged dataset size: {len(all_valid_data)}")


Processing datasets...
[BB1] Kept 0 valid pairs. Removed 0 outliers/invalid.
[BB2] Kept 3 valid pairs. Removed 137 outliers/invalid.

Total clean, merged dataset size: 3


---

# 8. Dataset Splitting & Export
We now randomly shuffle our clean dataset and split it into Train (70%), Val (20%), and Test (10%), exporting the images and properly normalized `.txt` labels to the `unified_dataset` folder structured for YOLO.

# 9. Model Selection
Based on the project requirements for efficient yet accurate detection of blueberries in potentially dense clusters, we select **YOLOv8** (specifically the `yolov8s.pt` variant) as our primary model for object detection and classification.

YOLOv8 is chosen for its:
- Real-time detection capabilities
- High accuracy on small objects (crucial for blueberries)
- Ease of integration with Python and PyTorch

In [19]:
!pip install ultralytics
from ultralytics import YOLO
import os

# Initialize the YOLOv8s model with pretrained weights
# This will download 'yolov8s.pt' if it doesn't already exist
model = YOLO('yolov8s.pt')

print("Model Selection and Initialization Complete!")
model.info()


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Model Selection and Initialization Complete!
YOLOv8s summary: 129 layers, 11,166,560 parameters, 0 gradients, 28.8 GFLOPs


(129, 11166560, 0, 28.816844800000002)

# 10. Model Training
We now initiate the training process. To achieve high accuracy and prevent overfitting, we use the following configuration:
- **Epochs**: 100
- **Image Size**: 640
- **Patience**: 20 (Early stopping if no improvement is seen)
- **Cosine LR**: Enabled for smoother convergence
- **Data Augmentation**: Enabled by default in YOLOv8

The training progress and results will be saved in the `runs/detect/train/` directory.

In [20]:
import torch

# Select device automatically: GPU if available, else CPU
device = 0 if torch.cuda.is_available() else 'cpu'
print(f"Using device: {'GPU' if device == 0 else 'CPU'}")

# Define training parameters
results = model.train(
    data='unified_dataset/data.yaml', 
    epochs=100, 
    imgsz=640, 
    batch=16, 
    patience=20, 
    optimizer='auto', 
    lr0=0.01, 
    cos_lr=True, 
    val=True, 
    device=device, 
    name='blueberry_yolo_model'
)

print("Training Complete!")

Using device: CPU
Ultralytics 8.4.21  Python-3.12.7 torch-2.10.0+cpu CPU (13th Gen Intel Core i5-13420H)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=unified_dataset/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=blueberry_yolo_model2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mas

RuntimeError: Dataset 'unified_dataset/data.yaml' error  'unified_dataset/data.yaml' does not exist